<font color='red'><b>**WARNING**</b></font> <br/>
어떠한 사유로도 임의로 복사, 촬영, 녹음, 복제, 보관, 전송하거나 허가 받지 않은 저장매체를 이용한 보관, 제3자에게 누설, 공개 또는 사용하는 등의 무단 사용 및 불법 배포 시 법적 조치를 받을 수 있습니다. <br/>

<div style="text-align: right; color: #7f8c8d; font-size: 0.9em; margin-top: 20px;">
📝 Author: 박사홍 (Sahong Pak)</br>
📧 Contact: sahong.pak@gmail.com</br>
📌 Version: v2.0</br>
📅 Last Updated: 2026-03-12</br>
</div>

# 학습 내용
>이번 장에서는 <strong>StateGraph와 ReAct 패턴(StateGraph & ReAct Pattern)</strong>에 대해 학습합니다.
>Direct 패턴과 ReAct 패턴의 차이를 이해하고, StateGraph로 Agent 그래프를 직접 학습해봅시다.

# Direct 패턴 (Direct Pattern)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">한 번의 LLM 호출</mark>로 도구 사용 여부를 결정하고 바로 응답하는 단순 패턴입니다.

실제 업무에서 AI가 처리해야 하는 작업은 단순 Q&A를 넘어섭니다. "3일 전에 산 상품 환불 가능해?"라는 질문 하나에도 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">정책 검색 → 날짜 계산 → 조건 판단 → 답변 생성</mark>의 여러 단계가 필요합니다. <strong>StateGraph</strong>는 이런 복잡한 워크플로우를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">노드(처리 단계)와 엣지(흐름 방향)로 구조화</mark>하여 관리할 수 있게 하고, <strong>ReAct 패턴</strong>은 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Reasoning(추론) + Acting(행동)</mark>의 합성어로, LLM이 생각하고 → 도구를 사용하고 → 결과를 관찰하고 → 다시 생각하는 루프를 구현합니다. 이 패턴이 없으면 다단계 작업을 코드로 하드코딩해야 하지만, StateGraph + ReAct를 사용하면 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LLM이 스스로 판단하며 단계를 진행</mark>할 수 있습니다.</br>
이 내용을 학습하기 전에 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">Agent 구성요소</mark>(LLM, Tool, Planning, Memory의 역할, Ch.4-2-1_001 참고)와 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">LangGraph 기본</mark>(StateGraph, 노드, 엣지의 개념과 <code>add_node</code>, <code>add_edge</code> 사용법)을 먼저 이해하면 좋습니다.

In [ ]:
# TODO 1: 도구 호출 여부를 판단하는 분기 함수를 정의하여 마지막 메시지에 도구 호출이 있으면 "tools", 없으면 종료를 반환하세요. 상태 그래프로 Direct 패턴 그래프를 구성하고 (agent→tools→종료), "환불 정책 알려줘"로 실행하여 결과를 출력하세요.

def should_continue(state):
    """도구 호출 필요 여부 판단"""
    last_message = state["messages"][-1]
    if last_message.tool_calls:
        return "tools"
    return END

# Direct 패턴 그래프
graph = StateGraph(MessagesState)
graph.add_node("agent", call_model)
graph.add_node("tools", tool_node)

graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue)
graph.add_edge("tools", END)  # 도구 실행 후 바로 종료

app = graph.compile()
result = app.invoke({"messages": [HumanMessage(content="환불 정책 알려줘")]})
print(f"메시지 수: {len(result['messages'])}")
print(f"최종 응답: {result['messages'][-1].content[:80]}...")

# ReAct 패턴 (Reasoning + Acting)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">추론(Reason)과 행동(Act)을 반복</mark>하며 복잡한 질문을 단계적으로 해결합니다.

In [ ]:
# TODO 2: ReAct 패턴 그래프를 구성하세요. Direct 패턴과 동일하지만, 도구 실행 후 다시 agent로 돌아가도록 엣지를 변경하세요. "3일 전에 산 상품 환불 가능해?"로 실행하여 결과를 출력하세요.

graph = StateGraph(MessagesState)
graph.add_node("agent", call_model)
graph.add_node("tools", tool_node)

graph.add_edge(START, "agent")
graph.add_conditional_edges("agent", should_continue)
graph.add_edge("tools", "agent")  # 도구 실행 후 다시 추론

app = graph.compile()
result = app.invoke({"messages": [HumanMessage(content="3일 전에 산 상품 환불 가능해?")]})
print(f"메시지 수: {len(result['messages'])}")
print(f"최종 응답: {result['messages'][-1].content[:100]}...")

## Direct vs ReAct 비교

<table style="width:100%">
  <thead>
    <tr>
      <th style="text-align:center">비교 항목</th>
      <th>Direct</th>
      <th>ReAct</th>
    </tr>
  </thead>
  <tbody>
    <tr><td style="text-align:center">도구 호출</td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">1회</mark></td><td><mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">반복 가능</mark></td></tr>
    <tr><td style="text-align:center">복잡한 질문</td><td>한계 있음</td><td>단계적 해결</td></tr>
    <tr><td style="text-align:center">도구→이후</td><td>END (종료)</td><td>agent (재추론)</td></tr>
    <tr><td style="text-align:center">비용</td><td>낮음</td><td>상대적으로 높음</td></tr>
  </tbody>
</table>

## conditional_edges (조건부 분기)
> <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">상태에 따라 다른 노드로 분기</mark>하는 핵심 메커니즘입니다.

In [ ]:
# TODO 3: 조건부 엣지를 사용하여 "agent" 노드에서 분기 함수에 따라 "tools" 또는 종료로 분기하도록 설정하세요.

graph.add_conditional_edges(
    "agent",           # 소스 노드
    should_continue    # 분기 함수 (→ "tools" 또는 END)
)

💡ReAct의 핵심
> 도구 실행 결과를 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">다시 LLM에 피드백</mark>하여 추가 추론이 가능합니다.
> "환불 정책 검색 → 조건 확인 → 환불 처리" 같은 다단계 작업에 적합합니다.

💡ReAct 무한 루프 방지
> ReAct는 이론적으로 <mark style="background-color:#FFF9C4; padding:2px 6px; border-radius:4px;">무한 반복</mark>할 수 있습니다.
> `recursion_limit`을 설정하여 최대 반복 횟수를 제한해야 합니다.